# Standard Du-DNN Training

Uses pre-split train/test CSV files from `1. Dataset/Python_DataProcess/`.

**Data strategy:**
1. Load the fixed training CSV (105,116 rows) and test CSV (5,533 rows).
2. At each iteration, sample the first `N` rows from the training CSV (2000, 4000, 6000, …).
3. Split those `N` rows 80/20 into a training set and a validation set.
4. Train a fresh Du-DNN; track the best model by validation loss (early stopping).
5. Evaluate the best model on the fixed test set.
6. Stop when test R² ≥ 0.90 or all iterations are exhausted.

In [ ]:
from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Configuration

Hyperparameters are fixed to the standard Du-DNN values from Table 2 of the paper.

In [ ]:
GLOBAL_SEED = 42
TARGET = 'Du'

DATA_DIR = Path('../1. Dataset/Python_DataProcess')
TRAIN_CSV = DATA_DIR / 'RandomPushover_DL_Total_Train_new80.csv'
TEST_CSV  = DATA_DIR / 'RandomPushover_DL_Total_Test_new80.csv'

# Samples added per iteration
STEP_SIZE = 2000
# Train/val split ratio within each sampled subset
TRAIN_VAL_RATIO = 0.80
# Stop when test R² reaches this threshold
R2_THRESHOLD = 0.90
# Maximum number of iterations
MAX_ITERATIONS = 40

INPUT_COLS = [
    'B', 'D', 'H', 'B_num', 'D_num', 'H_num',
    'colWidth', 'beamDepth', 'beamRat', 'Asc', 'Asb',
    't', 'cover1', 'cover2', 'Ec', 'nu_c', 'fc',
    'fcuRat', 'eps_cu', 'Es', 'nu_s', 'fsy', 'eta'
]

HYPERPARAMS = {
    'neurons_per_layer': [256, 128, 64, 64, 64, 64],
    'batch_size': 128,
    'weight_decay': 2.0,
    'dropout_rate': 0.30,
    'batch_norm': True,
    'optimizer': 'AdamW',
    'initial_lr': 1.0e-5,
    'lr_scheduler': 'ReduceLROnPlateau',
    'lr_factor': 0.2,
    'lr_patience': 10,
    'max_epochs': 3000,
    'early_stopping_patience': 500,
    'loss_function': 'MSE',
}

print(f'Target: {TARGET}')
print(f'Train CSV: {TRAIN_CSV.resolve()}')
print(f'Test CSV:  {TEST_CSV.resolve()}')
print(f'Step size: {STEP_SIZE} samples per iteration')
print(f'Train/val split within each subset: {TRAIN_VAL_RATIO:.0%} / {1-TRAIN_VAL_RATIO:.0%}')
print(f'Hyperparameters: {HYPERPARAMS}')

## Load Pre-Split Data

- **Training pool** (`RandomPushover_DL_Total_Train_new80.csv`): 105,116 rows — subsets are drawn from here.
- **Test set** (`RandomPushover_DL_Total_Test_new80.csv`): 5,533 rows — held out for final evaluation only.

Rows with `Du ≤ 0` are filtered out before anything else.

In [ ]:
df_train_pool = pd.read_csv(TRAIN_CSV)
df_test       = pd.read_csv(TEST_CSV)

# Filter invalid rows
df_train_pool = df_train_pool[df_train_pool[TARGET] > 0].reset_index(drop=True)
df_test       = df_test[df_test[TARGET] > 0].reset_index(drop=True)

print(f'Training pool : {len(df_train_pool):,} rows (after Du > 0 filter)')
print(f'Test set      : {len(df_test):,} rows (after Du > 0 filter)')

X_test_raw = df_test[INPUT_COLS].values.astype(np.float32)
y_test_raw = df_test[TARGET].values.astype(np.float32)

# Build iteration sample sizes: 2000, 4000, ..., up to training pool size
max_samples = min(len(df_train_pool), MAX_ITERATIONS * STEP_SIZE)
sample_sizes = list(range(STEP_SIZE, max_samples + 1, STEP_SIZE))
print(f'\nIterations planned: {len(sample_sizes)}')
print(f'Sample sizes: {sample_sizes[:5]} ... {sample_sizes[-1]:,}')

## Model and Training Utilities

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class RCFDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y.reshape(-1, 1))

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class DNNModel(nn.Module):
    def __init__(self, n_inputs, neurons, dropout_rate, use_batch_norm):
        super().__init__()
        layers = []
        in_dim = n_inputs
        for width in neurons:
            layers.append(nn.Linear(in_dim, width))
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(width))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            in_dim = width
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def make_loaders(X_train, y_train, X_val, y_val, batch_size):
    pin = device.type == 'cuda'
    train_loader = DataLoader(
        RCFDataset(X_train, y_train),
        batch_size=batch_size,
        shuffle=True,
        pin_memory=pin,
        num_workers=0,
    )
    val_loader = DataLoader(
        RCFDataset(X_val, y_val),
        batch_size=batch_size * 4,
        shuffle=False,
        pin_memory=pin,
        num_workers=0,
    )
    return train_loader, val_loader


def make_test_loader(X_test, y_test, batch_size):
    pin = device.type == 'cuda'
    return DataLoader(
        RCFDataset(X_test, y_test),
        batch_size=batch_size * 4,
        shuffle=False,
        pin_memory=pin,
        num_workers=0,
    )


def fmt_time(seconds):
    seconds = int(seconds)
    if seconds < 60:
        return f'{seconds}s'
    if seconds < 3600:
        return f'{seconds // 60}m {seconds % 60}s'
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f'{h}h {m}m {s}s'


def print_epoch_log_header():
    print()
    print(
        f"    {'Epoch':>6}  "
        f"{'Train MSE':>12}  "
        f"{'Val MSE':>12}  "
        f"{'Train R2':>9}  "
        f"{'Val R2':>8}  "
        f"{'LR':>10}  "
        f"{'Time':>12}"
    )
    print(
        f"    {'-' * 6}  "
        f"{'-' * 12}  "
        f"{'-' * 12}  "
        f"{'-' * 9}  "
        f"{'-' * 8}  "
        f"{'-' * 10}  "
        f"{'-' * 12}"
    )


def print_epoch_log_row(epoch, train_mse, val_mse, train_r2, val_r2, cumulative_time, current_lr):
    print(
        f"    {epoch:6d}  "
        f"{train_mse:12.6f}  "
        f"{val_mse:12.6f}  "
        f"{train_r2:9.4f}  "
        f"{val_r2:8.4f}  "
        f"{current_lr:10.2e}  "
        f"{fmt_time(cumulative_time):>12}"
    )

## Train One Iteration

For each iteration:
1. Take the first `N` rows from the training pool.
2. Split 80/20 into train and validation sets.
3. Fit a `MinMaxScaler` on the training features only; apply to val and test.
4. Train with early stopping on **validation loss**; keep the best model state.
5. Evaluate the best model on the **fixed test set** (never seen during training).

In [ ]:
def train_one_iteration(iteration, n_samples, df_pool, X_test_raw, y_test_raw):
    set_seed(GLOBAL_SEED)

    # --- Sample subset from the training pool ---
    subset = df_pool.iloc[:n_samples]
    X_subset = subset[INPUT_COLS].values.astype(np.float32)
    y_subset = subset[TARGET].values.astype(np.float32)

    # --- 80/20 train/val split ---
    X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(
        X_subset, y_subset,
        train_size=TRAIN_VAL_RATIO,
        random_state=GLOBAL_SEED + iteration,
        shuffle=True,
    )

    # --- Scale inputs (fit on train only) ---
    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
    X_val   = scaler.transform(X_val_raw).astype(np.float32)
    X_test  = scaler.transform(X_test_raw).astype(np.float32)

    y_train = y_train_raw.astype(np.float32)
    y_val   = y_val_raw.astype(np.float32)
    y_test  = y_test_raw.astype(np.float32)

    train_loader, val_loader = make_loaders(
        X_train, y_train, X_val, y_val, HYPERPARAMS['batch_size']
    )
    test_loader = make_test_loader(X_test, y_test, HYPERPARAMS['batch_size'])

    model = DNNModel(
        n_inputs=len(INPUT_COLS),
        neurons=HYPERPARAMS['neurons_per_layer'],
        dropout_rate=HYPERPARAMS['dropout_rate'],
        use_batch_norm=HYPERPARAMS['batch_norm'],
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=HYPERPARAMS['initial_lr'],
        weight_decay=HYPERPARAMS['weight_decay'],
    )
    # Reduce LR by factor 0.2 when val loss stops improving for 10 epochs (paper Section 4.1)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=HYPERPARAMS['lr_factor'],
        patience=HYPERPARAMS['lr_patience'],
    )

    best_val_loss = float('inf')
    best_state = None
    no_improve = 0
    train_losses, val_losses, epoch_times = [], [], []
    train_start = time.perf_counter()

    print_epoch_log_header()

    for epoch in range(HYPERPARAMS['max_epochs']):
        epoch_start = time.perf_counter()

        # --- Train ---
        model.train()
        train_running = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device).squeeze(-1)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            train_running += loss.item() * len(xb)
        train_loss = train_running / len(train_loader.dataset)
        train_losses.append(train_loss)

        # --- Validate ---
        model.eval()
        val_running = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device).squeeze(-1)
                val_running += criterion(model(xb), yb).item() * len(xb)
        val_loss = val_running / len(val_loader.dataset)
        val_losses.append(val_loss)
        epoch_times.append(time.perf_counter() - epoch_start)

        # --- Step LR scheduler on validation loss ---
        scheduler.step(val_loss)

        # --- Periodic logging ---
        if (epoch + 1) % 500 == 0:
            with torch.no_grad():
                pred_tr = model(torch.from_numpy(X_train).to(device)).cpu().numpy()
                pred_vl = model(torch.from_numpy(X_val).to(device)).cpu().numpy()
            train_r2 = r2_score(y_train_raw, pred_tr)
            val_r2   = r2_score(y_val_raw, pred_vl)
            current_lr = optimizer.param_groups[0]['lr']
            print_epoch_log_row(
                epoch + 1, train_loss, val_loss, train_r2, val_r2,
                time.perf_counter() - train_start,
                current_lr,
            )

        # --- Early stopping on validation loss ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= HYPERPARAMS['early_stopping_patience']:
                print(f'    Early stopping at epoch {epoch + 1}.')
                break

    # --- Load best model ---
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()

    # --- Final metrics on train, val, and test ---
    with torch.no_grad():
        pred_train = model(torch.from_numpy(X_train).to(device)).cpu().numpy()
        pred_val   = model(torch.from_numpy(X_val).to(device)).cpu().numpy()
        pred_test  = model(torch.from_numpy(X_test).to(device)).cpu().numpy()

    return {
        'model': model,
        'scaler': scaler,
        'iteration': iteration,
        'n_samples': n_samples,
        'train_rows': len(y_train),
        'val_rows': len(y_val),
        'test_rows': len(y_test),
        'train_mse': mean_squared_error(y_train_raw, pred_train),
        'val_mse':   mean_squared_error(y_val_raw,   pred_val),
        'test_mse':  mean_squared_error(y_test_raw,  pred_test),
        'train_r2':  r2_score(y_train_raw, pred_train),
        'val_r2':    r2_score(y_val_raw,   pred_val),
        'test_r2':   r2_score(y_test_raw,  pred_test),
        'train_losses': train_losses,
        'val_losses': val_losses,
        'epoch_times': epoch_times,
        'y_train_true': y_train_raw,
        'y_val_true':   y_val_raw,
        'y_test_true':  y_test_raw,
        'pred_train': pred_train,
        'pred_val':   pred_val,
        'pred_test':  pred_test,
    }

## Run Iterative Training

In [ ]:
SEP = '=' * 70
print('\n' + SEP)
print(f'  TRAINING FOR OUTPUT: {TARGET}')
print(SEP)

all_results = []
best_result = None
best_test_r2 = -float('inf')

for iteration, n_samples in enumerate(sample_sizes, start=1):
    n_train = int(n_samples * TRAIN_VAL_RATIO)
    n_val   = n_samples - n_train
    iter_start = time.perf_counter()

    print(
        f"\n  ITERATION {iteration:3d} | SUBSET={n_samples:,} "
        f"| TRAIN={n_train:,} | VAL={n_val:,} "
        f"| TEST={len(y_test_raw):,} (fixed)"
    )

    result = train_one_iteration(
        iteration, n_samples, df_train_pool, X_test_raw, y_test_raw
    )
    iter_elapsed = time.perf_counter() - iter_start
    all_results.append(result)

    print(
        f"\n  ITER {iteration:3d} DONE | subset={n_samples:,} | "
        f"train={result['train_rows']:,} | val={result['val_rows']:,} | test={result['test_rows']:,} | "
        f"train MSE={result['train_mse']:.6f} | val MSE={result['val_mse']:.6f} | "
        f"test MSE={result['test_mse']:.6f} | "
        f"train R2={result['train_r2']:.4f} | val R2={result['val_r2']:.4f} | "
        f"test R2={result['test_r2']:.4f} | "
        f"iter time={fmt_time(iter_elapsed)}"
    )

    if result['test_r2'] > best_test_r2:
        best_test_r2 = result['test_r2']
        best_result = result
        print(f"\n  *** New best model: test R2={best_test_r2:.4f}")
    else:
        print(f"\n  --- No improvement (best so far: test R2={best_test_r2:.4f})")

    if result['test_r2'] >= R2_THRESHOLD:
        print(f"\n  >>> R2 threshold {R2_THRESHOLD} reached at subset n={n_samples:,}. Stopping.")
        break

print(f"\n" + SEP)
print(
    f"  Best model -- iteration={best_result['iteration']} | subset={best_result['n_samples']:,} | "
    f"train={best_result['train_rows']:,} | val={best_result['val_rows']:,} | "
    f"test={best_result['test_rows']:,} | "
    f"train R2={best_result['train_r2']:.4f} | val R2={best_result['val_r2']:.4f} | "
    f"test R2={best_result['test_r2']:.4f}"
)
print(SEP)

## Results Table

In [ ]:
results_df = pd.DataFrame([
    {
        'iteration':  r['iteration'],
        'n_samples':  r['n_samples'],
        'train_rows': r['train_rows'],
        'val_rows':   r['val_rows'],
        'test_rows':  r['test_rows'],
        'train_mse':  r['train_mse'],
        'val_mse':    r['val_mse'],
        'test_mse':   r['test_mse'],
        'train_r2':   r['train_r2'],
        'val_r2':     r['val_r2'],
        'test_r2':    r['test_r2'],
        'epochs_run': len(r['train_losses']),
        'epoch_time_total_s': sum(r['epoch_times']),
    }
    for r in all_results
])

display(results_df)

if len(results_df) > 0:
    print('\nBest row (by test R2):')
    display(results_df.loc[[results_df['test_r2'].idxmax()]])